# Knife Baseline v1 — CUDA handoff
W4 visual review 승인 후에만 full training을 실행한다. 설정은 development baseline이며 최종 연구 파라미터가 아니다.

In [ ]:
REVIEW_APPROVED = False  # 팀 검수 완료 후에만 True
assert REVIEW_APPROVED, 'Stop: local W4 visual review approval is required.'


In [ ]:
import torch
assert torch.cuda.is_available(), 'CUDA GPU is unavailable'
print(torch.__version__, torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
REPO = Path('/content/edge-threat-response')
if not REPO.exists():
    !git clone --recurse-submodules https://github.com/jaydenkim197/edge-threat-response.git {REPO}
%cd {REPO}
!python -m pip install -r requirements/ml-smoke.txt
!python -m pip install -e . --no-deps


In [ ]:
!etr-dataset audit --registry configs/datasets/legacy.json --repo-root . --output-dir data/work/legacy-audit --fail-on never
!etr-dataset plan-split --manifest data/work/legacy-audit/manifest.jsonl --output-dir data/work/legacy-development-split --ratios train=0.7,val=0.15,test=0.15 --seed 20260915
!etr-dataset materialize-knife-yolo --manifest data/work/legacy-development-split/planned-manifest.jsonl --registry configs/datasets/legacy.json --repo-root . --output-dir data/processed/knife-legacy-development-v1 --link-mode hardlink


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT = Path('/content/drive/MyDrive/edge-threat-response/runs')
OUTPUT.mkdir(parents=True, exist_ok=True)
!etr-train --config configs/training/cuda-baseline-v1.json --data data/processed/knife-legacy-development-v1/data.yaml --dataset-manifest data/processed/knife-legacy-development-v1/materialized-manifest.jsonl --output-dir {OUTPUT} --preflight-only --require-cuda


In [ ]:
# 실행 전 preflight.json의 status=passed를 확인한다.
!etr-train --config configs/training/cuda-baseline-v1.json --data data/processed/knife-legacy-development-v1/data.yaml --dataset-manifest data/processed/knife-legacy-development-v1/materialized-manifest.jsonl --output-dir {OUTPUT}
